# Idea 1 of the ideas planner: wider scale coverage (0.80-1.05 x headings -5, 0, +5 = 15 queries) on clean and height-metadata +20 % photos
The same 60 official photos as Test_Iteration_9. Reading offline in Idea1_Scale_Coverage_Analysis.ipynb.


In [1]:
import os, sys, time
from pathlib import Path
import numpy as np, pandas as pd
ROOT = Path.cwd()
while not (ROOT / "single_photo_pipeline.py").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT)); sys.path.insert(0, str(ROOT / "src")); os.chdir(ROOT)
DATA = os.environ.get("GATE_DATA", "official")
POOL_PHOTOS = int(os.environ.get("POOL_PHOTOS", "500"))
PHOTOS_FILE = os.environ["PHOTOS_FILE"]
BASE_TAG = os.environ["BASE_TAG"]
from train.checks.perturb import perturbed_item
KINDS = [k for k in os.environ.get("KINDS", "clean,height20").split(",") if k]
OUT = ROOT / "tests/Test_Iteration_7/results"
print(DATA, BASE_TAG, KINDS)

official official60_scale15 ['clean', 'height20']


In [2]:
from train.checks.rich_dump import RichDumpPipeline, pack_photo
from image_processing.thermal import load_thermal_frame
pipeline = RichDumpPipeline(DATA, "edge_zncc", semantic_layer=True, semantic_decision=True, gamma=1.0, k_peaks=50, m_fine=10,
                            matcher="loftr_minima", min_inliers=8, ransac_pixels=None, matcher_confidence=0.22, satellite="new",
                            map_margin_m=None, loftr_coarse_threshold=0.21, dino_weight=None,
                            headings=[-5.0, 0.0, 5.0], scale_factors=[0.80, 0.85, 0.90, 1.0, 1.05])
t0 = time.time(); pipeline.setup(POOL_PHOTOS, 7); print("setup %.0f s" % (time.time() - t0))
test_set = pipeline.test_set
indices = sorted(int(i) for i in pd.read_csv(PHOTOS_FILE).photo)
len(indices)

level 3: 0.85 m/pixel, picture 16378 x 16005, map coverage 100%
level 4: 1.20 m/pixel, picture 11581 x 11317, map coverage 100%
level 5: 1.70 m/pixel, picture 8189 x 8002, map coverage 100%
level 6: 2.40 m/pixel, picture 5790 x 5658, map coverage 100%
level 7: 3.39 m/pixel, picture 4094 x 4001, map coverage 100%


map store ready, took 113 s


/home/voidness/Storage/Synced Folders/InterIIT Tech 15/venv-geoloc/lib/python3.10/site-packages/kornia/feature/lightglue.py:44: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  @torch.cuda.amp.custom_fwd(cast_inputs=torch.float32)


/home/voidness/Storage/Synced Folders/InterIIT Tech 15/venv-geoloc/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


setup 129 s


60

In [3]:
import torch
summary = []
for kind in KINDS:
    DUMP = OUT / "rich_dump" / f"{BASE_TAG}_{kind}"; DUMP.mkdir(parents=True, exist_ok=True)
    done = {int(p.stem.split("_")[1]) for p in DUMP.glob("photo_*.npz")}
    start = time.time(); failures = []
    for index in [i for i in indices if i not in done]:
        item = dict(test_set.items[index]) if kind == "clean" else perturbed_item(test_set.items[index], kind, str(OUT / "perturbed_photos" / BASE_TAG), load_thermal_frame)
        try:
            found = None
            for attempt in range(3):
                try:
                    found = pipeline.run_queries(item, test_set.priors[index]); break
                except torch.cuda.OutOfMemoryError:
                    torch.cuda.empty_cache(); time.sleep(5)
            if found is None:
                raise MemoryError("GPU out of memory three times")
            arrays = pack_photo(item, test_set.truths[index], test_set.sigmas[index], found["level"], found["queries"], min_keep_confidence=0.15)
            arrays["height"] = float(test_set.items[index]["rel_height"]); arrays["yaw"] = float(test_set.items[index]["yaw"])   # keep the true attitude
            np.savez_compressed(DUMP / ("photo_%d.npz" % index), **arrays)
        except Exception as error:
            failures.append((index, repr(error)[:200])); print(kind, "photo", index, "failed", repr(error)[:200])
    pd.DataFrame(failures, columns=["photo", "error"]).to_csv(DUMP / "failures.csv", index=False)
    summary.append(dict(kind=kind, photos=len(list(DUMP.glob("photo_*.npz"))), failed=len(failures), seconds=round(time.time() - start)))
    print(summary[-1], flush=True)
pd.DataFrame(summary)

{'kind': 'clean', 'photos': 60, 'failed': 0, 'seconds': 615}


{'kind': 'height20', 'photos': 60, 'failed': 0, 'seconds': 537}


,kind,photos,failed,seconds
0,clean,60,0,615
1,height20,60,0,537
